# Customer Churn Prediction (Telco) - Final Exam Project
## Full ML Pipeline: Data → EDA → Models → Evaluation


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [ ]:
# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Basic cleaning
df.drop('customerID', axis=1, inplace=True)

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Target encoding
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df.head()

## EDA (Exploratory Data Analysis)

In [ ]:
print(df.shape)
print(df.isnull().sum())

plt.figure(figsize=(5,3))
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.show()

plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## Encoding categorical variables

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()

## Train/Test Split + Scaling

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    results[name] = (acc, f1)
    print(name)
    print('Accuracy:', acc)
    print('F1-score:', f1)
    print(classification_report(y_test, pred))
    print('-'*40)


## Model Comparison

In [ ]:
results_df = pd.DataFrame(results, index=['Accuracy', 'F1-score']).T
display(results_df)

results_df.plot(kind='bar', figsize=(8,5))
plt.title('Model Comparison')
plt.show()

## Final Model (Best) + Confusion Matrix

In [ ]:
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix - Random Forest')
plt.show()

## Conclusion
- Best model: Random Forest
- Problem solved: Customer churn prediction
- Business value: Identify customers likely to leave and take preventive actions